# Week 9 - Morfologi Citra
Pengolahan Citra dan Visi Komputer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from skimage import filters
from skimage.morphology import skeletonize, thin

# lokasi folder gambar di Drive
PATH = '/content/drive/MyDrive/pcvk_2026/pertemuan 9/'


def tampil(gambar, judul):
    # tampilkan beberapa gambar berdampingan
    for k in range(len(gambar)):
        plt.subplot(1, len(gambar), k + 1)
        plt.imshow(gambar[k], cmap='gray')
        plt.title(judul[k])
        plt.xticks([]), plt.yticks([])
    plt.show()

## Persiapan citra plat nomor
Citra dibaca grayscale lalu di-threshold supaya jadi citra biner (0 dan 255).

In [ ]:
plat = cv2.imread(PATH + 'plat nomer.jpg', 0)
ret, biner = cv2.threshold(plat, 127, 255, cv2.THRESH_BINARY)

tampil([plat, biner], ['Citra Awal', 'Citra Biner'])

## 3. Dilasi cross 3 x 3 (tanpa library)

In [ ]:
def dilasi_citra(F, SE):
    p, q = F.shape
    a, b = SE.shape[0] // 2, SE.shape[1] // 2

    imgD = np.zeros((p, q), dtype=np.uint8)

    for i in range(a, p - a):
        for j in range(b, q - b):
            temp = F[i-a:i+a+1, j-b:j+b+1]
            imgD[i, j] = np.max(temp[SE == 1])
    return imgD


SED = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
print(SED)

imgD = dilasi_citra(biner, SED)
tampil([plat, imgD], ['Citra Awal', 'Citra Hasil Dilasi'])

**Analisis:** dilasi membuat huruf putih jadi lebih tebal dan celah kecil di antara huruf mulai tertutup.

## 4. Dilasi cross 3 x 3 (dengan library OpenCV)

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
dilasi = cv2.dilate(biner, kernel, iterations=1)
print(kernel)

tampil([plat, dilasi], ['Citra Awal', 'Citra Hasil Dilasi'])

# cek apakah hasilnya sama dengan fungsi buatan sendiri (tepi diabaikan)
print('Sama dengan no. 3:', np.array_equal(dilasi[1:-1, 1:-1], imgD[1:-1, 1:-1]))

**Analisis:** hasilnya sama dengan no. 3 (dicek dengan `np.array_equal`), jadi fungsi buatan sendiri sudah benar.

## 5. Erosi square 5 x 5 (tanpa library)

In [ ]:
def erosi_citra(F, SE):
    m, n = F.shape
    a, b = SE.shape[0] // 2, SE.shape[1] // 2

    imgE = np.zeros((m, n), dtype=np.uint8)

    for i in range(a, m - a):
        for j in range(b, n - b):
            temp = F[i-a:i+a+1, j-b:j+b+1]
            imgE[i, j] = np.min(temp[SE == 1])
    return imgE


SE = np.ones((5, 5), dtype=np.uint8)
imgE = erosi_citra(biner, SE)

tampil([plat, imgE], ['Citra Awal', 'Citra Hasil Erosi'])

**Analisis:** erosi menipiskan objek putih. Goresan huruf yang tipis banyak yang terputus karena SE 5 x 5 lebih lebar dari goresannya.

## 6. Erosi square 5 x 5 (dengan library OpenCV)

In [ ]:
kernel = np.ones((5, 5), np.uint8)
erosion = cv2.erode(biner, kernel, iterations=1)

tampil([plat, erosion], ['Citra Awal', 'Citra Hasil Erosi'])

print('Sama dengan no. 5:', np.array_equal(erosion[2:-2, 2:-2], imgE[2:-2, 2:-2]))

**Analisis:** sama dengan hasil no. 5, hanya beda cara (fungsi sendiri vs `cv2.erode`).

## 7. Opening square 3 x 3
Opening = erosi dulu, baru dilasi.

In [ ]:
# tanpa library
SE = np.ones((3, 3), dtype=np.uint8)
imgO = dilasi_citra(erosi_citra(biner, SE), SE)

tampil([plat, imgO], ['Citra Awal', 'Citra Hasil Opening'])

In [ ]:
# dengan library
kernel = np.ones((3, 3), np.uint8)
openn = cv2.morphologyEx(biner, cv2.MORPH_OPEN, kernel)

tampil([plat, biner, openn], ['Citra Awal', 'Citra Biner', 'Citra Hasil Opening'])

**Analisis:** opening (erosi lalu dilasi) menghilangkan bintik putih kecil, bentuk huruf utama hampir tidak berubah.

## 8. Closing square 3 x 3
Closing = dilasi dulu, baru erosi.

In [ ]:
# tanpa library
SE = np.ones((3, 3), dtype=np.uint8)
imgC = erosi_citra(dilasi_citra(biner, SE), SE)

tampil([plat, imgC], ['Citra Awal', 'Citra Hasil Closing'])

In [ ]:
# dengan library
kernel = np.ones((3, 3), np.uint8)
closs = cv2.morphologyEx(biner, cv2.MORPH_CLOSE, kernel)

tampil([plat, biner, closs], ['Citra Awal', 'Citra Biner', 'Citra Hasil Closing'])

**Analisis:** closing (dilasi lalu erosi) menutup lubang kecil berwarna hitam di dalam objek, ukuran huruf hampir sama seperti semula.

## 9. Dilasi dengan SE Cross 3 x 3 dan 5 x 5

In [ ]:
for w in (3, 5):
    SED = cv2.getStructuringElement(cv2.MORPH_CROSS, (w, w))
    print('SED = Cross', w)
    print(SED)

    imgD = dilasi_citra(biner, SED)
    tampil([plat, imgD], ['Citra Awal', 'Citra Hasil Dilasi'])

**Analisis:** cross 5 x 5 menebalkan objek lebih banyak daripada cross 3 x 3, dan arah penebalannya mengikuti bentuk tanda plus.

## 10. Dilasi dengan SE Circular 3 x 3 dan 5 x 5

In [ ]:
for w in (3, 5):
    SED = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (w, w))
    print('SED = Circle', w)
    print(SED)

    imgD = dilasi_citra(biner, SED)
    tampil([plat, imgD], ['Citra Awal', 'Citra Hasil Dilasi'])

**Analisis:** untuk ukuran 3 x 3, `MORPH_ELLIPSE` di OpenCV menghasilkan matriks yang sama dengan cross 3 x 3 (lihat matriks yang di-print), jadi hasilnya sama dengan no. 9. Pada ukuran 5 x 5 bentuknya lebih membulat.

## 11. Erosi dengan SE Rectangle 3 x 5 dan 5 x 7
Ukuran ditulis tinggi x lebar.

In [ ]:
for tinggi, lebar in ((3, 5), (5, 7)):
    SE = np.ones((tinggi, lebar), dtype=np.uint8)
    print('SE = Rectangle', tinggi, 'x', lebar)

    imgE = erosi_citra(biner, SE)
    tampil([plat, imgE], ['Citra Awal', 'Citra Hasil Erosi'])

**Analisis:** makin besar SE, makin banyak bagian huruf yang hilang. Rectangle 5 x 7 menyisakan lebih sedikit daripada 3 x 5.

## 12. Erosi dengan SE Line Vertikal 3 dan 5
File j.png tidak ada, jadi dipakai noisy_text.png (teks putih dengan noise).

In [ ]:
teks = cv2.imread(PATH + 'noisy_text.png', 0)
ret, biner_teks = cv2.threshold(teks, 127, 255, cv2.THRESH_BINARY)

for k in (3, 5):
    SE = np.ones((k, 1), dtype=np.uint8)
    print('SE = Line Vertikal', k)

    imgE = erosi_citra(biner_teks, SE)
    tampil([teks, imgE], ['Citra Awal', 'Citra Hasil Erosi'])

**Analisis:** line vertikal hanya mengikis ke arah atas-bawah, jadi noise yang sendirian sebagian hilang tapi yang bergerombol masih ada dan tepi huruf jadi bergerigi. Hasil k = 3 dan k = 5 mirip. Sebagai pembanding, lihat opening + closing di bawah.

### Tambahan: opening lalu closing pada noisy_text.png
Untuk membandingkan dengan line vertikal di atas.

In [ ]:
kernel = np.ones((3, 3), np.uint8)
buka = cv2.morphologyEx(biner_teks, cv2.MORPH_OPEN, kernel)
tutup = cv2.morphologyEx(buka, cv2.MORPH_CLOSE, kernel)

tampil([biner_teks, buka, tutup], ['Citra Biner', 'Opening', 'Opening + Closing'])

## 13. Top-hat 3 x 3

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

tophat_img = cv2.morphologyEx(plat, cv2.MORPH_TOPHAT, kernel)

tampil([plat, tophat_img], ['Citra Awal', 'Citra Hasil TOP HAT'])

**Analisis:** top-hat (citra asli dikurangi opening) hanya menyisakan detail terang yang kecil, seperti tepi huruf. Bagian yang luas menjadi gelap.

## 14. Black-hat 3 x 3
Black-hat = hasil closing dikurangi citra asli.

In [ ]:
kucing = cv2.imread(PATH + 'kitten01 (1).jpg')
kucing = cv2.cvtColor(kucing, cv2.COLOR_BGR2GRAY)

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
blackhat = cv2.morphologyEx(kucing, cv2.MORPH_BLACKHAT, kernel)

# cek manual: closing - citra asli
closing = cv2.morphologyEx(kucing, cv2.MORPH_CLOSE, kernel)
print('Sama dengan closing - citra asli:', np.array_equal(blackhat, closing - kucing))

tampil([kucing, blackhat], ['Citra Awal', 'Citra Hasil Black Hat'])

In [ ]:
# sebagai pembanding, kernel yang lebih besar (13 x 5)
kernel2 = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
blackhat2 = cv2.morphologyEx(kucing, cv2.MORPH_BLACKHAT, kernel2)

tampil([kucing, blackhat2], ['Citra Awal', 'Black Hat 13 x 5'])

**Analisis:** black-hat (closing dikurangi citra asli) menyisakan detail gelap yang kecil, misalnya garis bulu, mata, dan bayangan. Kernel yang lebih besar menangkap struktur gelap yang lebih besar.

## 15. Skeleton pada citra biner

In [ ]:
lily = cv2.imread(PATH + 'lily.jpg', 0)

binary = lily > filters.threshold_triangle(lily)
skeleton = skeletonize(binary)

plt.figure(figsize=(12, 6))
tampil([binary, skeleton], ['Citra Biner', 'Skeleton'])

**Analisis:** karena memakai `>`, yang bernilai True adalah latar putihnya, jadi skeleton yang terbentuk adalah rangka latar di sekeliling bunga, bukan rangka bunganya.

## 16. Skeleton pada citra hasil inversi
Tanda `>` diganti `<`, jadi bunga yang bernilai True.

In [ ]:
binary_inv = lily < filters.threshold_triangle(lily)
skeleton_inv = skeletonize(binary_inv)

plt.figure(figsize=(12, 6))
tampil([binary_inv, skeleton_inv], ['Citra Biner (inversi)', 'Skeleton (inversi)'])

**Analisis:** setelah diinversi (`<`) bunga menjadi objek, sehingga skeleton ada di dalam bunga mengikuti kelopaknya. Tepi bunga yang tidak rata membuat banyak cabang kecil.

## 17. Thinning dengan morphology.thin()

In [ ]:
out_skeletonize = skeletonize(binary_inv)
out_thin = thin(binary_inv)

plt.figure(figsize=(15, 5))
tampil([binary_inv, out_skeletonize, out_thin], ['Citra Biner', 'Skeleton', 'Thin'])

**Analisis:** `thin()` dan `skeletonize()` sama-sama menghasilkan garis setebal 1 piksel yang mengikuti bentuk bunga, tetapi pola garis keduanya tidak persis sama.